In [ ]:
from __future__ import annotations

import argparse
import csv
import json
import os
import random
import shutil
import subprocess
import sys
import time
import urllib.request
from collections import Counter
from pathlib import Path
from statistics import mean, median

In [ ]:
!pip install rdkit

In [ ]:
# RDKit
!pip install -q rdkit
# Check GPU
!nvidia-smi

In [ ]:
!pip uninstall -y tensorflow tensorflow-gpu tensorflow-cpu

In [ ]:

from __future__ import annotations

import argparse
import csv
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
import urllib.request
from collections import Counter
from pathlib import Path
from statistics import mean, median


# =========================
# User settings
# =========================

TARGET_UNIQUE = 1_200_000          # 3 lakh unique molecules
SAMPLE_BATCH_SIZE = 50_000       # batch sampling is safer than one huge sampling run
MAX_TOTAL_SAMPLES = 1_200_000    # stop if diversity collapses before target is reached

TL_EPOCHS = 10                   # transfer learning on all 1446 active compounds
TL_BATCH_SIZE = 64

RL_STEPS = 500                   # increase if score is still improving
RL_BATCH_SIZE = 64


MAX_RL_REFERENCE_SMILES = 1_500


HIGH_ACTIVITY_TANIMOTO_THRESHOLD = 0.7


TARGET_HIGH_ACTIVITY = 5_000
UNIQUE_OVERSHOOT_MULTIPLIER = 1.5

SEED = 42
PROCESSOR_IF_GPU = "cu126"

ENABLE_TENSORBOARD = False

BASE_TEMPERATURE = 1.0
MAX_TEMPERATURE = 1.6
TEMPERATURE_STEP = 0.15
LOW_GAIN_FRACTION = 0.05        # a batch is "low gain" if unique growth < 5% of batch size
LOW_GAIN_BATCHES_BEFORE_STOP = 4  # stop sampling after this many consecutive low-gain batches

REINVENT_GIT_REF = "main"
REINVENT_REPO = "https://github.com/MolecularAI/REINVENT4.git"
ZENODO_RECORD_URL = "https://zenodo.org/api/records/15641297"

BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
REPO_DIR = BASE_DIR / "REINVENT4"
RUN_DIR = BASE_DIR / "reinvent4_akt1_unique_generation"
PRIOR_DIR = BASE_DIR / "reinvent_priors"


import signal as _signal_module

CAPTURE_DIR_NAME = "_command_output_captures"


_ALLOWED_SMILES_CHARS = set(
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "abcdefghijklmnopqrstuvwxyz"
    "0123456789"
    "()[]=#-+@/\\.%*:"
)


def _first_disallowed_char(smi: str) -> str | None:
    """Return the first character in smi that falls outside REINVENT's SMILES
    alphabet, or None if the string is clean. Returning the actual character (not
    just a bool) lets callers report exactly what was found, including its Unicode
    code point, which is essential for diagnosing look-alike characters that are
    visually indistinguishable from a normal one in a terminal or log file.
    """
    for ch in smi:
        if ch not in _ALLOWED_SMILES_CHARS:
            return ch
    return None


def print_log_tail(cwd: Path | None, log_name: str | None, lines: int = 160) -> bool:
    """Print the tail of a tool-generated log file, if it exists.

    Returns True if anything was printed, so callers know whether they still
    need to fall back to the raw captured stdout.
    """
    if not cwd or not log_name:
        return False

    log_path = cwd / log_name
    if not log_path.exists():
        return False

    print(f"\n--- Last {lines} lines from {log_path} ---")
    try:
        content = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        for line in content[-lines:]:
            print(line)
    except Exception as exc:
        print(f"Could not read log file {log_path}: {exc}")
        return False
    return True


def signal_name(returncode: int) -> str | None:
    """subprocess returns a negative returncode when the child was killed by a
    signal. Translate that into a readable name (e.g. -9 -> SIGKILL, usually an
    out-of-memory kill; -11 -> SIGSEGV, a native crash inside a compiled dependency).
    """
    if returncode >= 0:
        return None
    try:
        return _signal_module.Signals(-returncode).name
    except ValueError:
        return f"signal {-returncode}"


def run_command(cmd: list[str], cwd: Path | None = None, log_name: str | None = None) -> str:
    """Run a command and, critically, ALWAYS persist its full combined output to a
    file on disk, not just to the notebook cell. Colab's output panel is scrollable
    and it's easy to copy only the visible tail and miss the real error sitting
    above it. Writing to disk means the real error is never lost, regardless of
    what got copied into a chat window or how long the tool's own log file took to
    get created.
    """
    print("\n>>>", " ".join(str(part) for part in cmd))
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    capture_dir = (cwd or BASE_DIR) / CAPTURE_DIR_NAME
    capture_dir.mkdir(exist_ok=True)
    safe_name = (log_name or f"{cmd[0]}_{int(time.time())}").replace("/", "_")
    capture_path = capture_dir / f"{safe_name}.full_output.txt"
    capture_path.write_text(result.stdout or "(no output captured)", encoding="utf-8")

    print(result.stdout)
    print(f"Exit code: {result.returncode}")

    if result.returncode != 0:
        sig = signal_name(result.returncode)
        if sig:
            hint = (
                " This usually means Colab killed the process for running out of RAM/VRAM."
                if sig == "SIGKILL"
                else " This usually means a native crash inside a compiled dependency, not a Python-level error."
            )
            print(f"Process was killed by {sig}.{hint}")

        print(f"\nFull captured output (this is the source of truth, more reliable than the cell above): {capture_path}")
        found_tool_log = print_log_tail(cwd, log_name)
        if not found_tool_log:
            print(
                f"No {log_name or 'tool log'} was found, which usually means the process failed "
                "before it finished setting up its own logging. Check the captured output file above "
                "for the actual error, it's the full stdout/stderr regardless of when the crash happened."
            )
        raise RuntimeError(
            f"Command failed (exit {result.returncode}): {' '.join(cmd)}\n"
            f"Full output saved to: {capture_path}"
        )

    return result.stdout


def check_dependency_conflicts(cwd: Path) -> None:
    """Run `pip check` right after install and surface any conflicts loudly.

    REINVENT4's own requirements can collide with packages Colab preinstalls
    (e.g. pandas version pins used by google-colab, cudf-cu12, dask-cudf-cu12).
    These conflicts don't always fail the install step itself, they can instead
    surface later as an obscure crash the first time a conflicting package's code
    path actually runs. Better to know about them up front.
    """
    result = subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.returncode != 0:
        print("\n" + "=" * 70)
        print("DEPENDENCY CONFLICT WARNING (pip check found issues)")
        print("=" * 70)
        print(result.stdout)
        print(
            "These conflicts may or may not cause a crash, but if a REINVENT4 run "
            "fails early with little or no output, this is the first place to look.\n"
        )
    else:
        print("pip check: no dependency conflicts found.")


def detect_device() -> tuple[str, str]:
    """Detect GPU availability without crashing on machines that lack nvidia-smi.

    subprocess.run raises FileNotFoundError (not a nonzero return code) when the
    executable itself isn't found. On a CPU-only machine that exception used to kill
    the whole script before it ever got to the CPU fallback below.
    """
    try:
        gpu_check = subprocess.run(
            ["nvidia-smi", "-L"],
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )
        if gpu_check.returncode == 0:
            return "cuda:0", PROCESSOR_IF_GPU
    except FileNotFoundError:
        pass
    return "cpu", "cpu"


def remove_conflicting_tensorflow() -> None:
    """Uninstall the standalone tensorflow package, if present.

    Colab preinstalls tensorflow built against its own bundled CUDA/cuDNN version.
    reinvent only requires `tensorboard`, not full `tensorflow`, but something in
    the lightning/torch.utils.tensorboard import chain can still eagerly import
    tensorflow if it's on the path. When that happens in the same process where
    torch has already initialized CUDA with the fresh cu126 libraries pip just
    installed, the two CUDA runtimes collide and the process dies with SIGSEGV
    (exit code -11) the instant TF's native library finishes loading, before any
    Python-level error or reinvent log ever gets written.

    tensorboard is explicitly designed to work without tensorflow installed (it
    falls back to its own lightweight summary writer), so removing tensorflow
    here costs nothing functionally and removes the crash trigger.
    """
    print("\nRemoving standalone tensorflow package to prevent a CUDA-init SIGSEGV "
          "when it shares a process with torch's freshly installed CUDA libraries...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "tensorflow", "tensorflow-gpu", "tensorflow-cpu"],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)


def install_reinvent(processor: str) -> str:
    if not REPO_DIR.exists():
        run_command(["git", "clone", "--depth", "1", "--branch", REINVENT_GIT_REF, REINVENT_REPO, str(REPO_DIR)])
    else:
        print(f"Using existing REINVENT4 repo: {REPO_DIR}")

    run_command([sys.executable, "install.py", processor], cwd=REPO_DIR)
    check_dependency_conflicts(REPO_DIR)
    remove_conflicting_tensorflow()

    try:
        commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=str(REPO_DIR),
            text=True,
        ).strip()
    except Exception:
        commit = "unknown"

    print("REINVENT4 commit:", commit)
    return commit


def upload_or_get_csv(path_arg: str | None) -> Path:
    if path_arg:
        path = Path(path_arg)
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    try:
        from google.colab import files
    except Exception as exc:
        raise RuntimeError("Pass --csv when not running inside Google Colab.") from exc

    print("\nUpload your CSV file containing a SMILES column.")
    uploaded = files.upload()
    csv_name = next(iter(uploaded.keys()))
    return BASE_DIR / csv_name


def read_smiles_csv(csv_path: Path) -> tuple[list[str], str, list[str]]:
    with open(csv_path, "r", encoding="utf-8-sig", errors="replace", newline="") as handle:
        sample = handle.read(4096)
        handle.seek(0)
        try:
            dialect = csv.Sniffer().sniff(sample)
        except Exception:
            dialect = csv.excel

        reader = csv.DictReader(handle, dialect=dialect)
        rows = list(reader)

    if not rows:
        raise ValueError("CSV file is empty or could not be read.")

    columns = list(rows[0].keys())
    possible_cols = ["SMILES", "smiles", "Smiles", "canonical_smiles", "Canonical_SMILES", "smile"]
    smiles_col = next((col for col in possible_cols if col in columns), columns[0])

    raw_smiles = []
    for row in rows:
        value = row.get(smiles_col, "")
        if value is not None:
            value = str(value).strip()
            if value:
                raw_smiles.append(value)

    if not raw_smiles:
        raise ValueError("No SMILES found in selected CSV column.")

    print("Detected columns:", columns)
    print("Using SMILES column:", smiles_col)
    return raw_smiles, smiles_col, columns


def canonicalize_input_smiles(raw_smiles: list[str]) -> tuple[list[str], int, int]:
    """Canonicalize and dedupe input SMILES.

    Returns (valid_unique_smiles, invalid_count, rejected_notation_count).

    Two defenses against the "RuntimeError: unknown token |" crash that used to kill
    transfer learning partway through epoch 1:

      1. If RDKit parses an input SMILES into a mol that carries enhanced-stereochemistry
         StereoGroups (from a CXSMILES suffix such as "|&1:5,10|" in the source data),
         those groups are cleared with mol.SetStereoGroups([]) before re-serializing.
         RDKit re-emits that "|...|" suffix on ANY mol that still has StereoGroups
         attached when you ask for isomeric SMILES.
      2. As a backstop, any canonical SMILES that still contains a character outside
         REINVENT's actual SMILES alphabet is dropped rather than written to the
         training/reference files, and counted separately so it's visible in the run
         summary. This is a WHITELIST, not a blacklist of specific characters: it also
         catches look-alike Unicode characters (e.g. a fullwidth or box-drawing bar that
         renders identically to '|' in a terminal but isn't the same code point), which
         a blacklist of specific known-bad characters would silently miss.
    """
    from rdkit import Chem

    valid = []
    invalid_count = 0
    rejected_notation_count = 0

    for smi in raw_smiles:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            invalid_count += 1
            continue

        if mol.GetStereoGroups():
            mol.SetStereoGroups([])

        canon = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

        bad_char = _first_disallowed_char(canon)
        if bad_char is not None:
            print(
                f"  Dropping input SMILES: unsupported character {bad_char!r} "
                f"(U+{ord(bad_char):04X}) in canonicalized SMILES: {canon!r}"
            )
            rejected_notation_count += 1
            continue

        valid.append(canon)

    return list(dict.fromkeys(valid)), invalid_count, rejected_notation_count


def write_smi(path: Path, smiles: list[str]) -> None:
    path.write_text("\n".join(smiles) + "\n", encoding="utf-8")


def validate_smi_file(path: Path) -> None:
    """Re-read a .smi file immediately after writing it and fail fast, with the exact
    line number and offending character, if anything outside REINVENT's SMILES alphabet
    is present.

    This exists as a second line of defense on top of the whitelist check inside
    canonicalize_input_smiles(). If that check somehow still misses something (e.g. a
    future code path writes to one of these files without going through
    canonicalize_input_smiles first), this turns what would otherwise be a deep, opaque
    crash inside torch's dataloader mid-training -- reporting only a tokenized character
    list, not which file or line it came from -- into an immediate, precise error before
    `reinvent` is ever invoked, including the character's Unicode code point so
    look-alike characters (e.g. a fullwidth vertical bar that's visually identical to
    '|' but a different code point) are unambiguous.
    """
    with open(path, "r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            smi = line.strip()
            if not smi:
                continue
            bad_char = _first_disallowed_char(smi)
            if bad_char is not None:
                raise ValueError(
                    f"{path} line {line_no} contains a character REINVENT's tokenizer "
                    f"cannot handle: {bad_char!r} (U+{ord(bad_char):04X}) in SMILES: "
                    f"{smi!r}. This should have been filtered during canonicalization; "
                    "please report this SMILES."
                )


def toml_string(value: Path | str) -> str:
    return json.dumps(str(value))


def split_train_validation(smiles: list[str]) -> tuple[list[str], list[str]]:
    """Split into transfer-learning train/validation sets.

    The previous version applied a 50-molecule floor and then a `len // 3` cap, and the
    cap could still undercut the floor for datasets in roughly [100, 150) molecules --
    e.g. a 120-molecule dataset would get `120 // 3 = 40` validation molecules instead of
    the intended 50, silently. This version caps only at the dataset size itself (leaving
    at least one molecule for training), so the 50-molecule floor genuinely holds for
    every dataset with >= 100 molecules, and the 200-molecule ceiling still applies for
    large datasets.
    """
    rng = random.Random(SEED)
    shuffled = list(smiles)
    rng.shuffle(shuffled)

    if len(shuffled) < 100:
        return shuffled, []

    val_size = int(round(len(shuffled) * 0.10))
    val_size = max(50, val_size)
    val_size = min(val_size, 200, len(shuffled) - 1)
    validation = shuffled[:val_size]
    train = shuffled[val_size:]
    return train, validation


def pick_diverse_references(smiles: list[str], max_refs: int) -> list[str]:
    if len(smiles) <= max_refs:
        return smiles

    from rdkit import Chem, DataStructs
    from rdkit.Chem import rdFingerprintGenerator
    from rdkit.SimDivFilters.rdSimDivPickers import MaxMinPicker

    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    fps = []
    kept_smiles = []

    for smi in smiles:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        fps.append(generator.GetFingerprint(mol))
        kept_smiles.append(smi)

    def distance(i: int, j: int) -> float:
        return 1.0 - DataStructs.TanimotoSimilarity(fps[i], fps[j])

    picker = MaxMinPicker()
    picked = list(picker.LazyPick(distance, len(fps), max_refs, seed=SEED))
    return [kept_smiles[i] for i in picked]


def download_prior() -> Path:
    PRIOR_DIR.mkdir(exist_ok=True)
    prior_file = PRIOR_DIR / "reinvent.prior"

    if prior_file.exists() and prior_file.stat().st_size > 0:
        print("Using existing prior:", prior_file)
        return prior_file

    print("Downloading reinvent.prior from Zenodo...")
    with urllib.request.urlopen(ZENODO_RECORD_URL, timeout=120) as response:
        data = json.loads(response.read().decode("utf-8"))

    target = None
    for file_info in data["files"]:
        if file_info["key"] == "reinvent.prior":
            target = file_info
            break

    if target is None:
        raise RuntimeError("Could not find reinvent.prior in Zenodo record.")

    urllib.request.urlretrieve(target["links"]["self"], prior_file)
    print("Prior file:", prior_file)
    print("Prior size MB:", round(prior_file.stat().st_size / 1024 / 1024, 2))
    return prior_file


def write_transfer_learning_toml(
    prior_file: Path,
    train_smi: Path,
    validation_smi: Path | None,
    device: str,
) -> Path:
    validation_line = f"validation_smiles_file = {toml_string(validation_smi)}" if validation_smi else ""
    tb_logdir_line = 'tb_logdir = "tb_logs_tl"' if ENABLE_TENSORBOARD else ""
    toml = f'''
run_type = "transfer_learning"
device = "{device}"
{tb_logdir_line}
json_out_config = "_transfer_learning.json"

[parameters]
input_model_file = {toml_string(prior_file)}
output_model_file = "tl_uploaded_1200plus.model"
smiles_file = {toml_string(train_smi)}
{validation_line}
num_epochs = {TL_EPOCHS}
batch_size = {TL_BATCH_SIZE}
sample_batch_size = 200
save_every_n_epochs = {TL_EPOCHS}
num_refs = 0
tb_isim = false
# Explicitly disabled. REINVENT4 re-serializes ("randomizes") every training SMILES
# with a fresh random atom ordering on every epoch when this is true (it defaults to
# true in REINVENT4's own reference notebook). That re-serialization happens inside
# reinvent's own RDKit calls, independent of the already-validated .smi files we hand
# it, and for a molecule with an unassigned/ambiguous stereocenter it can intermittently
# emit a CXSMILES-style token that was never in our input and isn't in the model's
# vocabulary, crashing training with "unknown token". Training directly on the fixed,
# pre-validated canonical strings avoids that entirely.
randomize_smiles = false
'''
    path = RUN_DIR / "transfer_learning_uploaded_smiles.toml"
    path.write_text(toml.strip() + "\n", encoding="utf-8")
    return path


def write_rl_toml(prior_file: Path, agent_file: Path, reference_smiles: list[str], device: str) -> Path:
    references = json.dumps(reference_smiles, indent=4)
    tb_logdir_line = 'tb_logdir = "tb_logs_rl"' if ENABLE_TENSORBOARD else ""
    toml = f'''
run_type = "staged_learning"
device = "{device}"
{tb_logdir_line}
json_out_config = "_rl_uploaded_smiles.json"

[parameters]
summary_csv_prefix = "rl_uploaded_smiles"
use_checkpoint = false
purge_memories = false

prior_file = {toml_string(prior_file)}
agent_file = {toml_string(agent_file)}

batch_size = {RL_BATCH_SIZE}
# NOTE: `unique_sequences` was removed from this block. The installed reinvent==4.8.24
# RLConfig pydantic model raises "Extra inputs are not permitted [type=extra_forbidden]"
# for parameters.unique_sequences -- this build's schema no longer accepts that key
# under [parameters] (it's a leftover from older REINVENT4 tutorial tomls). Generated
# molecules are still de-duplicated by REINVENT4 internally during staged learning and,
# separately, by this script's own dedup pass in sample_until_unique(), so dropping the
# now-invalid key does not change behavior -- it just stops asserting a knob this
# version doesn't expose here. If a future version reports the same extra_forbidden
# error for a different field name, delete/rename just that field.
randomize_smiles = true
tb_isim = false

[learning_strategy]
type = "dap"
sigma = 128
rate = 0.0001

[diversity_filter]
type = "IdenticalMurckoScaffold"
bucket_size = 50
minscore = 0.45
minsimilarity = 0.4
penalty_multiplier = 0.5

[[stage]]
chkpt_file = "rl_uploaded_smiles_final.chkpt"
termination = "simple"
max_score = 0.90
min_steps = 100
max_steps = {RL_STEPS}

[stage.scoring]
type = "geometric_mean"

[[stage.scoring.component]]
[stage.scoring.component.custom_alerts]

[[stage.scoring.component.custom_alerts.endpoint]]
name = "Remove unwanted groups"
weight = 1.0
params.smarts = [
    "[*;r{{8-17}}]",
    "[#8][#8]",
    "[#6;+]",
    "[#16][#16]",
    "[#7;!n][S;!$(S(=O)=O)]",
    "[#7;!n][#7;!n]",
    "C#C",
    "C(=[O,S])[O,S]"
]

[[stage.scoring.component]]
[stage.scoring.component.TanimotoSimilarity]

[[stage.scoring.component.TanimotoSimilarity.endpoint]]
name = "Similarity to diverse uploaded SMILES"
weight = 2.0
params.smiles = {references}
params.radius = 2
params.use_counts = true
params.use_features = true

[[stage.scoring.component]]
[stage.scoring.component.QED]

[[stage.scoring.component.QED.endpoint]]
name = "QED"
weight = 1.0

[[stage.scoring.component]]
[stage.scoring.component.MolecularWeight]

[[stage.scoring.component.MolecularWeight.endpoint]]
name = "Molecular weight"
weight = 1.0
transform.type = "double_sigmoid"
transform.low = 300.0
transform.high = 550.0
transform.coef_div = 550.0
transform.coef_si = 20.0
transform.coef_se = 20.0

[[stage.scoring.component]]
[stage.scoring.component.SlogP]

[[stage.scoring.component.SlogP.endpoint]]
name = "SlogP"
weight = 1.0
transform.type = "double_sigmoid"
transform.low = 1.0
transform.high = 5.0
transform.coef_div = 5.0
transform.coef_si = 20.0
transform.coef_se = 20.0

[[stage.scoring.component]]
[stage.scoring.component.TPSA]

[[stage.scoring.component.TPSA.endpoint]]
name = "TPSA"
weight = 1.0
transform.type = "double_sigmoid"
transform.low = 40.0
transform.high = 140.0
transform.coef_div = 140.0
transform.coef_si = 20.0
transform.coef_se = 20.0
'''
    path = RUN_DIR / "rl_uploaded_smiles.toml"
    path.write_text(toml.strip() + "\n", encoding="utf-8")
    return path


def write_sampling_toml(
    model_file: Path | str,
    output_name: str,
    batch_size: int,
    device: str,
    temperature: float = BASE_TEMPERATURE,
) -> Path:
    toml = f'''
run_type = "sampling"
device = "{device}"
json_out_config = "_sample_generated.json"

[parameters]
model_file = {toml_string(model_file)}
output_file = "{output_name}"
num_smiles = {batch_size}
unique_molecules = true
randomize_smiles = true
isomeric_smiles = true
temperature = {temperature}
'''
    path = RUN_DIR / "sample_generated.toml"
    path.write_text(toml.strip() + "\n", encoding="utf-8")
    return path


def detect_smiles_column(fieldnames: list[str] | None) -> str:
    if not fieldnames:
        raise ValueError("Generated CSV has no header.")
    possible = ["SMILES", "smiles", "Smiles", "canonical_smiles", "sampled_smiles"]
    return next((col for col in possible if col in fieldnames), fieldnames[0])


def iter_generated_smiles(csv_path: Path):
    with open(csv_path, "r", encoding="utf-8", errors="replace", newline="") as handle:
        reader = csv.DictReader(handle)
        smiles_col = detect_smiles_column(reader.fieldnames)
        for row in reader:
            smi = str(row.get(smiles_col, "")).strip()
            if smi:
                yield smi


def build_input_fingerprints(smiles: list[str]):
    """Precompute Morgan fingerprints for the full set of input active SMILES.

    Computed once, up front, and reused for every generated molecule's nearest-neighbor
    Tanimoto lookup in filter_high_activity_batch() -- recomputing 1446 fingerprints per
    generated molecule would be needlessly slow across a 300k-molecule sample.
    """
    from rdkit import Chem
    from rdkit.Chem import rdFingerprintGenerator

    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    fps = []
    for smi in smiles:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        fps.append(generator.GetFingerprint(mol))
    return generator, fps


def nearest_training_tanimoto(generator, input_fps: list, mol) -> float:
    """Max Tanimoto similarity between a generated molecule and any of the input
    actives -- i.e. the "nearest_training_tanimoto" value, same concept as the column
    already present in your original screening CSV.
    """
    from rdkit import DataStructs

    if not input_fps:
        return 0.0
    fp = generator.GetFingerprint(mol)
    return max(DataStructs.BulkTanimotoSimilarity(fp, input_fps))


def sample_until_unique(
    model_file: Path | str,
    device: str,
    input_smiles: set[str],
    input_fp_generator,
    input_fps: list,
) -> tuple[Path, Path, dict]:
    """Sample in batches until BOTH targets are reached, MAX_TOTAL_SAMPLES is hit, or
    diversity collapses:
      - TARGET_UNIQUE unique generated molecules overall
      - TARGET_HIGH_ACTIVITY of those with nearest-training Tanimoto >=
        HIGH_ACTIVITY_TANIMOTO_THRESHOLD (the "highly active" proxy filter)

    Changes from the original version:
      - Temperature is raised automatically after a low-gain batch, which usually
        recovers unique yield without any manual intervention.
      - After LOW_GAIN_BATCHES_BEFORE_STOP consecutive low-gain batches (even after
        raising temperature to its max), sampling stops early instead of burning
        through the full MAX_TOTAL_SAMPLES budget for little return. This early-stop
        is a safety valve against runaway compute, not a target-reached signal -- if it
        fires before TARGET_HIGH_ACTIVITY is met, stopped_early_reason explains why.
      - As a defensive backstop (the model shouldn't be able to generate CXSMILES
        characters since they're not in its vocabulary, but this costs nothing and
        guarantees a bad string can never reach the output file or the quality report).
      - NEW: every valid, unique molecule is also checked against the 1446 input actives
        via nearest_training_tanimoto(). Molecules clearing HIGH_ACTIVITY_TANIMOTO_THRESHOLD
        are written incrementally to a second CSV (high_activity_candidates.csv) with
        their similarity score, so you get a ready-to-use "highly active candidates" file
        without re-scanning the full unique-molecule output afterward.
    """
    from rdkit import Chem

    unique_output = RUN_DIR / f"unique_generated_{TARGET_UNIQUE}.csv"
    high_activity_output = RUN_DIR / "high_activity_candidates.csv"
    seen = set()
    high_activity_count = 0
    generated_rows = 0
    valid_rows = 0
    invalid_rows = 0
    total_sampled = 0
    batch_idx = 0
    last_unique_count = 0
    consecutive_low_gain = 0
    temperature = BASE_TEMPERATURE
    stopped_early_reason = None

    with open(unique_output, "w", encoding="utf-8", newline="") as out_handle, \
         open(high_activity_output, "w", encoding="utf-8", newline="") as ha_handle:
        writer = csv.writer(out_handle)
        writer.writerow(["canonical_smiles"])
        ha_writer = csv.writer(ha_handle)
        ha_writer.writerow(["canonical_smiles", "nearest_training_tanimoto"])

        while (
            (len(seen) < TARGET_UNIQUE or high_activity_count < TARGET_HIGH_ACTIVITY)
            and len(seen) < TARGET_UNIQUE * UNIQUE_OVERSHOOT_MULTIPLIER
            and total_sampled < MAX_TOTAL_SAMPLES
        ):
            batch_idx += 1
            batch_name = f"sample_batch_{batch_idx:04d}.csv"
            write_sampling_toml(model_file, batch_name, SAMPLE_BATCH_SIZE, device, temperature)
            batch_log = f"sample_batch_{batch_idx:04d}.log"
            run_command(
                ["reinvent", "-l", batch_log, "sample_generated.toml"],
                cwd=RUN_DIR,
                log_name=batch_log,
            )

            batch_csv = RUN_DIR / batch_name
            if not batch_csv.exists():
                raise FileNotFoundError(f"Expected sampling output not found: {batch_csv}")

            batch_new = 0
            batch_new_high_activity = 0
            for smi in iter_generated_smiles(batch_csv):
                generated_rows += 1
                mol = Chem.MolFromSmiles(smi)
                if mol is None:
                    invalid_rows += 1
                    continue
                if mol.GetStereoGroups():
                    mol.SetStereoGroups([])
                can = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
                if _first_disallowed_char(can) is not None:
                    invalid_rows += 1
                    continue
                valid_rows += 1
                if can in seen:
                    continue
                seen.add(can)
                writer.writerow([can])
                batch_new += 1

                similarity = nearest_training_tanimoto(input_fp_generator, input_fps, mol)
                if similarity >= HIGH_ACTIVITY_TANIMOTO_THRESHOLD:
                    ha_writer.writerow([can, round(similarity, 4)])
                    high_activity_count += 1
                    batch_new_high_activity += 1

                if len(seen) >= TARGET_UNIQUE and high_activity_count >= TARGET_HIGH_ACTIVITY:
                    break

            out_handle.flush()
            ha_handle.flush()
            total_sampled += SAMPLE_BATCH_SIZE
            gain = len(seen) - last_unique_count
            last_unique_count = len(seen)
            print(
                f"Unique molecules: {len(seen):,}/{TARGET_UNIQUE:,}; new this batch: "
                f"{batch_new:,}; gain: {gain:,}; high-activity candidates: "
                f"{high_activity_count:,}/{TARGET_HIGH_ACTIVITY:,} (+{batch_new_high_activity:,} "
                f"this batch); temperature: {temperature:.2f}"
            )

            if batch_idx >= 3 and gain < SAMPLE_BATCH_SIZE * LOW_GAIN_FRACTION:
                consecutive_low_gain += 1
                if temperature < MAX_TEMPERATURE:
                    temperature = round(min(temperature + TEMPERATURE_STEP, MAX_TEMPERATURE), 2)
                    print(f"Low yield detected. Raising sampling temperature to {temperature:.2f}.")
                if consecutive_low_gain >= LOW_GAIN_BATCHES_BEFORE_STOP:
                    stopped_early_reason = (
                        f"Stopped after {consecutive_low_gain} consecutive low-gain batches "
                        f"even at max temperature ({MAX_TEMPERATURE}). Diversity has likely "
                        "collapsed; consider reducing RL steps or diversity filter penalty. "
                        f"High-activity candidates at stop: {high_activity_count:,}/"
                        f"{TARGET_HIGH_ACTIVITY:,}."
                    )
                    print(f"\n{stopped_early_reason}")
                    break
            else:
                consecutive_low_gain = 0

    if high_activity_count < TARGET_HIGH_ACTIVITY and stopped_early_reason is None:
        if len(seen) >= TARGET_UNIQUE * UNIQUE_OVERSHOOT_MULTIPLIER:
            stopped_early_reason = (
                f"Stopped at {len(seen):,} unique molecules ({UNIQUE_OVERSHOOT_MULTIPLIER}x "
                f"TARGET_UNIQUE) with only {high_activity_count:,}/{TARGET_HIGH_ACTIVITY:,} "
                "high-activity candidates found. The high-activity yield per batch was too "
                "low to justify sampling further past this cap. Consider lowering "
                "HIGH_ACTIVITY_TANIMOTO_THRESHOLD, raising UNIQUE_OVERSHOOT_MULTIPLIER, or "
                "increasing RL_STEPS so the agent biases harder toward the active chemical space."
            )
            print(f"\n{stopped_early_reason}")
        else:
            print(
                f"\nReached MAX_TOTAL_SAMPLES ({MAX_TOTAL_SAMPLES:,}) with only "
                f"{high_activity_count:,}/{TARGET_HIGH_ACTIVITY:,} high-activity candidates found. "
                "Consider lowering HIGH_ACTIVITY_TANIMOTO_THRESHOLD, raising MAX_TOTAL_SAMPLES, "
                "or increasing RL_STEPS so the agent biases harder toward the active chemical space."
            )

    stats = {
        "target_unique": TARGET_UNIQUE,
        "unique_written": len(seen),
        "target_high_activity": TARGET_HIGH_ACTIVITY,
        "high_activity_tanimoto_threshold": HIGH_ACTIVITY_TANIMOTO_THRESHOLD,
        "high_activity_written": high_activity_count,
        "sampling_batches": batch_idx,
        "total_requested_samples": total_sampled,
        "generated_rows_read": generated_rows,
        "valid_rows_read": valid_rows,
        "invalid_rows_read": invalid_rows,
        "duplicates_removed": max(valid_rows - len(seen), 0),
        "novel_vs_input": sum(1 for smi in seen if smi not in input_smiles),
        "final_temperature": temperature,
        "stopped_early_reason": stopped_early_reason,
    }
    return unique_output, high_activity_output, stats


def property_summary(values: list[float]) -> dict[str, float | None]:
    if not values:
        return {"min": None, "median": None, "mean": None, "max": None}
    values = sorted(values)
    return {
        "min": round(values[0], 4),
        "median": round(median(values), 4),
        "mean": round(mean(values), 4),
        "max": round(values[-1], 4),
    }


def write_quality_report(unique_csv: Path, input_smiles: set[str], run_meta: dict) -> Path:
    """Compute property/scaffold/novelty stats for the generated set.

    NOTE ON SPEED: computing MolWt/logP/TPSA/QED/Murcko-scaffold in a pure-Python RDKit
    loop over every single unique molecule is the slowest part of the whole pipeline once
    sampling finishes -- QED in particular is one of RDKit's more expensive per-molecule
    calls. At 300k+ molecules this loop alone can silently run for many minutes with zero
    output, which is exactly what looked like a hang. Two fixes:
      1. DESCRIPTOR_SAMPLE_SIZE caps how many molecules get full descriptor/scaffold
         treatment, using the same random-subsample approach already used for the
         Tanimoto novelty check below, instead of processing the entire unique set.
      2. A progress print every DESCRIPTOR_PROGRESS_EVERY molecules, so a long report
         step is visibly moving instead of looking frozen.
    """
    from rdkit import Chem, DataStructs
    from rdkit.Chem import Crippen, Descriptors, QED, rdFingerprintGenerator
    from rdkit.Chem.Scaffolds import MurckoScaffold

    DESCRIPTOR_SAMPLE_SIZE = 50_000
    DESCRIPTOR_PROGRESS_EVERY = 10_000

    smiles = []
    with open(unique_csv, "r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            smi = row["canonical_smiles"].strip()
            if smi:
                smiles.append(smi)

    descriptor_limit = min(DESCRIPTOR_SAMPLE_SIZE, len(smiles))
    sample_rng = random.Random(SEED)
    descriptor_subset = (
        sample_rng.sample(smiles, descriptor_limit) if descriptor_limit < len(smiles) else smiles
    )
    if descriptor_limit < len(smiles):
        print(
            f"\nComputing property/scaffold stats on a random sample of "
            f"{descriptor_limit:,}/{len(smiles):,} unique molecules (full-set descriptor "
            "computation would be too slow at this scale)..."
        )
    else:
        print(f"\nComputing property/scaffold stats for all {len(smiles):,} unique molecules...")

    mw_values = []
    logp_values = []
    tpsa_values = []
    qed_values = []
    scaffold_counter = Counter()

    for idx, smi in enumerate(descriptor_subset, start=1):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        mw_values.append(Descriptors.MolWt(mol))
        logp_values.append(Crippen.MolLogP(mol))
        tpsa_values.append(Descriptors.TPSA(mol))
        qed_values.append(float(QED.qed(mol)))
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
        if scaffold:
            scaffold_counter[scaffold] += 1
        if idx % DESCRIPTOR_PROGRESS_EVERY == 0:
            print(f"  ...{idx:,}/{descriptor_limit:,} molecules processed")

    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    input_fps = []
    for smi in input_smiles:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            input_fps.append(generator.GetFingerprint(mol))

    # Random sample instead of the first N rows, so the novelty/similarity summary
    # isn't biased toward whatever the earliest sampling batches happened to produce.
    similarity_limit = min(50_000, len(smiles))
    similarity_subset = (
        sample_rng.sample(smiles, similarity_limit) if similarity_limit < len(smiles) else smiles
    )

    print(f"\nComputing nearest-training Tanimoto for a random sample of "
          f"{similarity_limit:,}/{len(smiles):,} unique molecules...")
    max_similarities = []
    for idx, smi in enumerate(similarity_subset, start=1):
        mol = Chem.MolFromSmiles(smi)
        if mol is None or not input_fps:
            continue
        fp = generator.GetFingerprint(mol)
        max_similarities.append(max(DataStructs.BulkTanimotoSimilarity(fp, input_fps)))
        if idx % DESCRIPTOR_PROGRESS_EVERY == 0:
            print(f"  ...{idx:,}/{similarity_limit:,} molecules processed")

    report = {
        **run_meta,
        "unique_file": str(unique_csv),
        "unique_count": len(smiles),
        "descriptor_sample_size": descriptor_limit,
        "novelty_fraction_vs_input": round(run_meta["novel_vs_input"] / max(len(smiles), 1), 4),
        "murcko_scaffold_count_in_sample": len(scaffold_counter),
        "top_20_murcko_scaffolds_in_sample": scaffold_counter.most_common(20),
        "molecular_weight": property_summary(mw_values),
        "slogp": property_summary(logp_values),
        "tpsa": property_summary(tpsa_values),
        "qed": property_summary(qed_values),
        "max_tanimoto_to_input_random_sample": property_summary(max_similarities),
        "max_tanimoto_sample_size": similarity_limit,
    }

    report_path = RUN_DIR / "generation_quality_report.json"
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("\nQuality report:", report_path)
    print(json.dumps(report, indent=2)[:4000])
    return report_path


def maybe_download_outputs(paths: list[Path]) -> None:
    try:
        from google.colab import files
    except Exception:
        return

    for path in paths:
        if path.exists():
            files.download(str(path))


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Improved REINVENT4 molecule generation workflow.")
    parser.add_argument("--csv", default=None, help="Path to input CSV. If omitted in Colab, upload prompt is used.")
    parser.add_argument("--target-unique", type=int, default=TARGET_UNIQUE, help="Number of unique molecules to write.")
    parser.add_argument("--sample-batch-size", type=int, default=SAMPLE_BATCH_SIZE, help="Sampling batch size.")
    parser.add_argument("--max-total-samples", type=int, default=MAX_TOTAL_SAMPLES, help="Maximum requested samples.")
    args, unknown = parser.parse_known_args()
    if unknown:
        print("Ignoring notebook/runtime arguments:", unknown)
    return args


def main() -> None:
    global TARGET_UNIQUE, SAMPLE_BATCH_SIZE, MAX_TOTAL_SAMPLES

    args = parse_args()
    TARGET_UNIQUE = args.target_unique
    SAMPLE_BATCH_SIZE = args.sample_batch_size
    MAX_TOTAL_SAMPLES = args.max_total_samples

    start = time.time()
    random.seed(SEED)

    device, processor = detect_device()
    print("Using device:", device)
    print("REINVENT processor:", processor)

    RUN_DIR.mkdir(exist_ok=True)

    commit = install_reinvent(processor)
    csv_path = upload_or_get_csv(args.csv)
    raw_smiles, smiles_col, columns = read_smiles_csv(csv_path)
    valid_smiles, invalid_count, rejected_notation_count = canonicalize_input_smiles(raw_smiles)

    if not valid_smiles:
        raise ValueError("No valid SMILES found in uploaded CSV.")

    print("Raw SMILES:", len(raw_smiles))
    print("Valid unique SMILES:", len(valid_smiles))
    print("Invalid SMILES:", invalid_count)
    print("Rejected (unsupported CXSMILES notation, e.g. enhanced stereo groups):", rejected_notation_count)

    train_smiles, validation_smiles = split_train_validation(valid_smiles)
    reference_smiles = pick_diverse_references(valid_smiles, MAX_RL_REFERENCE_SMILES)

    print("Transfer-learning train SMILES:", len(train_smiles))
    print("Transfer-learning validation SMILES:", len(validation_smiles))
    print("RL reference SMILES:", len(reference_smiles))

    input_smi = RUN_DIR / "uploaded_valid_smiles_all.smi"
    train_smi = RUN_DIR / "uploaded_train_smiles.smi"
    validation_smi = RUN_DIR / "uploaded_validation_smiles.smi"
    reference_smi = RUN_DIR / "uploaded_diverse_rl_references.smi"

    write_smi(input_smi, valid_smiles)
    write_smi(train_smi, train_smiles)
    if validation_smiles:
        write_smi(validation_smi, validation_smiles)
    write_smi(reference_smi, reference_smiles)


    validate_smi_file(input_smi)
    validate_smi_file(train_smi)
    if validation_smiles:
        validate_smi_file(validation_smi)
    validate_smi_file(reference_smi)

    prior_file = download_prior()

    tl_toml = write_transfer_learning_toml(
        prior_file=prior_file,
        train_smi=train_smi,
        validation_smi=validation_smi if validation_smiles else None,
        device=device,
    )
    run_command(
        ["reinvent", "-l", "transfer_learning.log", str(tl_toml.name)],
        cwd=RUN_DIR,
        log_name="transfer_learning.log",
    )

    tl_model = RUN_DIR / "tl_uploaded_1200plus.model"
    if not tl_model.exists():
        raise FileNotFoundError(f"Transfer-learning model not found: {tl_model}")

    rl_toml = write_rl_toml(
        prior_file=prior_file,
        agent_file=tl_model,
        reference_smiles=reference_smiles,
        device=device,
    )
    run_command(
        ["reinvent", "-l", "rl_uploaded_smiles.log", str(rl_toml.name)],
        cwd=RUN_DIR,
        log_name="rl_uploaded_smiles.log",
    )

    rl_model = RUN_DIR / "rl_uploaded_smiles_final.chkpt"
    if not rl_model.exists():
        raise FileNotFoundError(f"RL checkpoint not found: {rl_model}")

    input_set = set(valid_smiles)
    print(f"\nPrecomputing fingerprints for {len(valid_smiles):,} input actives "
          "(used for the nearest-training-Tanimoto high-activity filter)...")
    input_fp_generator, input_fps = build_input_fingerprints(valid_smiles)

    unique_csv, high_activity_csv, sampling_stats = sample_until_unique(
        rl_model, device, input_set, input_fp_generator, input_fps
    )

    run_meta = {
        "input_csv": str(csv_path),
        "input_smiles_column": smiles_col,
        "input_columns": columns,
        "raw_input_smiles": len(raw_smiles),
        "valid_unique_input_smiles": len(valid_smiles),
        "invalid_input_smiles": invalid_count,
        "rejected_cxsmiles_notation_smiles": rejected_notation_count,
        "tl_epochs": TL_EPOCHS,
        "rl_steps": RL_STEPS,
        "rl_reference_smiles": len(reference_smiles),
        "reinvent_git_ref": REINVENT_GIT_REF,
        "reinvent_commit": commit,
        "device": device,
        "elapsed_minutes": round((time.time() - start) / 60, 2),
        **sampling_stats,
    }
    report_path = write_quality_report(unique_csv, input_set, run_meta)

    print("\nDONE")
    print("Unique generated molecules:", unique_csv)
    print(
        f"Highly active candidates (Tanimoto to actives >= "
        f"{HIGH_ACTIVITY_TANIMOTO_THRESHOLD}): {high_activity_csv} "
        f"({sampling_stats['high_activity_written']:,}/{TARGET_HIGH_ACTIVITY:,} target)"
    )
    print("Quality report:", report_path)
    print("Run directory:", RUN_DIR)
    maybe_download_outputs([unique_csv, high_activity_csv, report_path, input_smi, reference_smi])


if __name__ == "__main__":
    main()